In [4]:
import pandas as pd
from scipy import stats

# Load cleaned files
reviews = pd.read_csv(
    r"C:\Users\saran\guvi project\cart2insights\data\cleaned\order_reviews_cleaned.csv"
)

orders = pd.read_csv(
    r"C:\Users\saran\guvi project\cart2insights\data\cleaned\orders_cleaned.csv"
)

# Convert required date columns
date_columns = [
    'order_purchase_timestamp',
    'order_delivered_customer_date',
    'order_estimated_delivery_date'
]

for col in date_columns:
    orders[col] = pd.to_datetime(orders[col])

# Create delivery status
orders['delivery_status'] = None

valid_delivery = (
    orders['order_delivered_customer_date'].notna() &
    orders['order_estimated_delivery_date'].notna()
)

orders.loc[
    valid_delivery &
    (orders['order_delivered_customer_date'] >
     orders['order_estimated_delivery_date']),
    'delivery_status'
] = 'Delayed'

orders.loc[
    valid_delivery &
    (orders['order_delivered_customer_date'] <=
     orders['order_estimated_delivery_date']),
    'delivery_status'
] = 'On Time'

# Merge reviews with delivery status
review_delivery = reviews.merge(
    orders[['order_id', 'delivery_status']],
    on='order_id',
    how='inner'
)

# Keep only On Time and Delayed
test_data = review_delivery[
    review_delivery['delivery_status'].isin(['On Time', 'Delayed'])
].copy()

# Make review score numeric
test_data['review_score'] = pd.to_numeric(
    test_data['review_score'],
    errors='coerce'
)

# Create the two groups
on_time_scores = test_data.loc[
    test_data['delivery_status'] == 'On Time',
    'review_score'
].dropna()

delayed_scores = test_data.loc[
    test_data['delivery_status'] == 'Delayed',
    'review_score'
].dropna()

# Average scores
print("On-time reviews:", len(on_time_scores))
print("Delayed reviews:", len(delayed_scores))

print("Average On-time review score:", on_time_scores.mean())
print("Average Delayed review score:", delayed_scores.mean())

# Levene's test
levene_result = stats.levene(
    on_time_scores,
    delayed_scores
)

print("Levene p-value:", levene_result.pvalue)

On-time reviews: 88658
Delayed reviews: 7701
Average On-time review score: 4.293577567732184
Average Delayed review score: 2.566549798727438
Levene p-value: 0.0


In [5]:
t_stat, p_value = stats.ttest_ind(
    delayed_scores,
    on_time_scores,
    equal_var=False
)

print("T-statistic:", t_stat)
print("P-value:", p_value)

T-statistic: -89.55081413351807
P-value: 0.0


In [6]:
import pandas as pd

order_items = pd.read_csv(
    r"C:\Users\saran\guvi project\cart2insights\data\cleaned\order_items_cleaned.csv"
)

products = pd.read_csv(
    r"C:\Users\saran\guvi project\cart2insights\data\cleaned\products_cleaned.csv"
)

category_translation = pd.read_csv(
    r"C:\Users\saran\guvi project\cart2insights\data\cleaned\product_category_name_translation_cleaned.csv"
)

print("Order items:", order_items.shape)
print("Products:", products.shape)
print("Categories:", category_translation.shape)

Order items: (112650, 7)
Products: (32951, 9)
Categories: (71, 2)


In [7]:
category_data = order_items.merge(
    products[['product_id', 'product_category_name']],
    on='product_id',
    how='left'
)

print(category_data.shape)
print(category_data[['product_id', 'product_category_name', 'price']].head())

(112650, 8)
                         product_id product_category_name   price
0  4244733e06e7ecb4970a6e2683c13e61            cool_stuff   58.90
1  e5f2d52b802189ee658865ca93d83a8f              pet_shop  239.90
2  c777355d18b72b67abbeef9df44fd0fd      moveis_decoracao  199.00
3  7634da152a4610f1595efa32f14722fc            perfumaria   12.99
4  ac6c3623068f30de03045865e4e10089    ferramentas_jardim  199.90


In [8]:
category_data = category_data.merge(
    category_translation,
    on='product_category_name',
    how='left'
)

print(category_data.shape)

print(
    category_data[
        ['product_category_name', 'product_category_name_english', 'price']
    ].head(10)
)

(112650, 9)
   product_category_name product_category_name_english   price
0             cool_stuff                    cool_stuff   58.90
1               pet_shop                      pet_shop  239.90
2       moveis_decoracao               furniture_decor  199.00
3             perfumaria                     perfumery   12.99
4     ferramentas_jardim                  garden_tools  199.90
5  utilidades_domesticas                    housewares   21.90
6              telefonia                     telephony   19.90
7     ferramentas_jardim                  garden_tools  810.00
8           beleza_saude                 health_beauty  145.95
9        livros_tecnicos               books_technical   53.99


In [9]:
category_summary = (
    category_data
    .groupby('product_category_name_english')['price']
    .agg(['count', 'mean'])
    .sort_values('mean', ascending=False)
)

category_summary.head(10)

,count,mean
product_category_name_english,,
computers,203,1098.340542
small_appliances_home_oven_and_coffee,76,624.285658
home_appliances_2,238,476.124958
agro_industry_and_commerce,212,342.124858
musical_instruments,680,281.616000
small_appliances,679,280.778468
fixed_telephony,264,225.693182
construction_tools_safety,194,208.992371
watches_gifts,5991,201.135984


In [10]:
category_groups = [
    group['price'].values
    for name, group in category_data.groupby(
        'product_category_name_english'
    )
    if len(group) >= 30
]

print("Number of categories used:", len(category_groups))

Number of categories used: 66


In [11]:
anova_result = stats.f_oneway(*category_groups)

print("F-statistic:", anova_result.statistic)
print("P-value:", anova_result.pvalue)

F-statistic: 212.4523949013122
P-value: 0.0


In [13]:
tukey_data = category_data[
    ['product_category_name_english', 'price']
].dropna()

print(tukey_data.shape)
print(tukey_data['product_category_name_english'].isna().sum())

(111023, 2)
0


In [14]:
from statsmodels.stats.multicomp import pairwise_tukeyhsd

tukey_result = pairwise_tukeyhsd(
    endog=tukey_data['price'],
    groups=tukey_data['product_category_name_english'],
    alpha=0.05
)

print(tukey_result)

                                     Multiple Comparison of Means - Tukey HSD, FWER=0.05                                     
                 group1                                  group2                  meandiff  p-adj    lower      upper   reject
-----------------------------------------------------------------------------------------------------------------------------
             agro_industry_and_commerce                        air_conditioning  -156.8556    0.0  -221.4273   -92.284   True
             agro_industry_and_commerce                                     art  -226.3228    0.0  -296.3277 -156.3178   True
             agro_industry_and_commerce                   arts_and_craftmanship  -266.5411    0.0   -421.213 -111.8692   True
             agro_industry_and_commerce                                   audio  -202.8707    0.0  -264.9179 -140.8236   True
             agro_industry_and_commerce                                    auto  -202.1673    0.0  -252.7111 -151.6235

In [15]:
order_payments = pd.read_csv(
    r"C:\Users\saran\guvi project\cart2insights\data\cleaned\order_payments_cleaned.csv"
)

orders = pd.read_csv(
    r"C:\Users\saran\guvi project\cart2insights\data\cleaned\orders_cleaned.csv"
)

print("Payments:", order_payments.shape)
print("Orders:", orders.shape)

Payments: (103886, 5)
Orders: (99441, 8)


In [16]:
payment_status = order_payments.merge(
    orders[['order_id', 'order_status']],
    on='order_id',
    how='inner'
)

print(payment_status.shape)
print(payment_status[['payment_type', 'order_status']].head())

(103886, 6)
  payment_type order_status
0  credit_card    delivered
1  credit_card    delivered
2  credit_card    delivered
3  credit_card    delivered
4  credit_card    delivered


In [17]:
contingency_table = pd.crosstab(
    payment_status['payment_type'],
    payment_status['order_status']
)

print(contingency_table)

order_status  approved  canceled  created  delivered  invoiced  processing  \
payment_type                                                                 
boleto               0        95        2      19191        67          70   
credit_card          2       444        3      74586       239         224   
debit_card           0         7        0       1486         6           2   
not_defined          0         3        0          0         0           0   
voucher              0       115        0       5493        13          23   

order_status  shipped  unavailable  
payment_type                        
boleto            209          150  
credit_card       851          446  
debit_card         22            6  
not_defined         0            0  
voucher            84           47  


In [19]:
chi2, p_value, dof, expected = stats.chi2_contingency(
    contingency_table
)

print("Chi-square statistic:", chi2)
print("P-value:", p_value)
print("Degrees of freedom:", dof)

Chi-square statistic: 677.0831369637666
P-value: 1.2047397488650834e-124
Degrees of freedom: 28
